**⭐ 1. What This Pattern Solves**

Removes duplicates from streaming or batch data within a defined window of time or rows.
Useful for:
Event streams where the same event may arrive multiple times.
Log aggregation to avoid double-counting.
Deduplicating transaction records within a period.
Ensures recent duplicates are filtered while older data is unaffected.

**⭐ 2. SQL Equivalent**

PARTITION BY defines the deduplication key.
ORDER BY defines which record to keep (e.g., latest).

In [0]:
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY user_id, event_type ORDER BY event_time DESC) AS rn
    FROM events
    WHERE event_time BETWEEN '2025-12-01' AND '2025-12-16'
)
SELECT *
FROM ranked
WHERE rn = 1;

**⭐ 3. Core Idea**

Keep the first (or last) occurrence of each key within a moving window; discard subsequent duplicates.

**⭐ 4. Template Code (MEMORIZE THIS)**

In [0]:
from collections import deque

window_size = 100  # rows or time-based
seen_keys = set()
window = deque()

for record in stream:
    key = (record['user_id'], record['event_type'])
    
    # remove old items
    while len(window) >= window_size:
        old_key = window.popleft()
        seen_keys.discard(old_key)
    
    if key not in seen_keys:
        seen_keys.add(key)
        window.append(key)
        process(record)

**⭐ 5. Detailed Example**

In [0]:
[
    {'user_id': 1, 'event': 'click'},
    {'user_id': 2, 'event': 'click'},
    {'user_id': 1, 'event': 'click'},
    {'user_id': 3, 'event': 'view'},
]

from collections import deque

window_size = 3
seen_keys = set()
window = deque()

for r in [
    {'user_id': 1, 'event': 'click'},
    {'user_id': 2, 'event': 'click'},
    {'user_id': 1, 'event': 'click'},
    {'user_id': 3, 'event': 'view'},
]:
    key = (r['user_id'], r['event'])
    while len(window) >= window_size:
        old_key = window.popleft()
        seen_keys.discard(old_key)
    if key not in seen_keys:
        seen_keys.add(key)
        window.append(key)
        print(r)

{'user_id': 1, 'event': 'click'}
{'user_id': 2, 'event': 'click'}
{'user_id': 3, 'event': 'view'}

**⭐ 6. Mini Practice Problems**

Deduplicate streaming logs by session_id within the last 50 events.
Remove repeated transactions per account_id in a 24-hour window.
Deduplicate IoT sensor readings per device_id within the last 10 readings.

**⭐ 7. Full Data Engineering Scenario**

Problem: Streaming click events arrive multiple times per user within seconds. We want only one click per user per 5-second window.

Expected Output: Deduplicated events stream with no repeated user clicks in 5 seconds.

Skeleton Solution:

In [0]:
from collections import deque
from time import time

window_sec = 5
seen = {}
window = deque()  # store (timestamp, key)

for event in event_stream:
    key = event['user_id']
    now = time()
    
    # expire old keys
    while window and now - window[0][0] > window_sec:
        _, old_key = window.popleft()
        seen.pop(old_key, None)
    
    if key not in seen:
        seen[key] = True
        window.append((now, key))
        process(event)

**⭐ 8. Time & Space Complexity**

Time: O(n) — each record is processed once; old keys removed in amortized O(1).
Space: O(k) — max size of window (k rows or events).

**⭐ 9. Common Pitfalls & Mistakes**

❌ Forgetting to remove old keys, leading to memory growth.
❌ Using only set without window, which deduplicates globally instead of within window.
✔ Correct approach: combine deque + set for sliding window deduplication.
❌ Ignoring key definition; duplicates must be defined per business logic.
❌ Not handling time-based windows in streaming, causing stale duplicates to remain.